In [ ]:
## import libraries

import os
import random
import zlib

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.cm import ScalarMappable
from matplotlib.colors import Normalize

from PIL import Image
from scipy.ndimage import gaussian_filter
from skimage.feature import canny
from pathlib import Path


import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path
from PIL import Image
from scipy.ndimage import gaussian_filter
from skimage.draw import line as draw_line

import json

import numpy as np
import matplotlib.pyplot as plt
from scipy import ndimage
from typing import Tuple, Optional

from PIL import Image
import statsmodels.formula.api as smf

In [ ]:
## define filepaths

project_path =  Path().resolve().parent
tables_path = os.path.join(project_path, "015_tables")
raw_data_path = os.path.join(project_path, "005_cleaned_data")
raw_data_path_targets = os.path.join(project_path, "007_targets")
viz_path = os.path.join(project_path, "020_visualizations")

print(project_path)

In [ ]:
# Automatically reload updated code from utils/*.py every time you run a cell
%load_ext autoreload
%autoreload 2

# Add the parent directory to the Python path (only needed if utils is not already importable)
import sys
from pathlib import Path
if str(project_path) not in sys.path:
    sys.path.append(str(project_path))

from utils.utils import *
from utils.detail_feature import *
from utils.viz import *

In [ ]:
# -------------------------------
# Load CSVs and register datasets
# (VizEx single-experiment version)
# -------------------------------

# Paths to CSV files
vizex_csv_path = os.path.join(tables_path, "vizex_features.csv")
vizex_targets_csv_path = os.path.join(tables_path, "vizex_targets_features.csv")

# Read CSVs
df_vizex = pd.read_csv(vizex_csv_path)
df_vizex_targets = pd.read_csv(vizex_targets_csv_path)

# Belt-and-braces: 015 now filters to cropped images at source, so this should be a
# no-op. Kept because .str.contains("fileType_cropped") is the correct test --
# .str.endswith("cropped.png") also matches "..._fileType_uncropped.png".
_n_before = len(df_vizex)
df_vizex = df_vizex[df_vizex["png_filename"].str.contains("fileType_cropped", na=False)].reset_index(drop=True)
if _n_before != len(df_vizex):
    print(f"WARNING: dropped {_n_before - len(df_vizex)} non-cropped rows -- re-run 015.")

# The feature tables must be the regenerated ones: correct filenames, no all-NaN columns.
_FEATURES = ["detail", "r_directionality", "theta_directionality", "brightness", "contrast"]
for _name, _d in [("vizex", df_vizex), ("vizex_targets", df_vizex_targets)]:
    for _c in _FEATURES:
        if _c not in _d.columns:
            raise KeyError(f"[{_name}] missing feature column '{_c}' -- re-run 015.")
        if _d[_c].isna().all():
            raise ValueError(f"[{_name}] feature column '{_c}' is entirely NaN -- re-run 015.")
print(f"vizex: {len(df_vizex)} cropped rows, {df_vizex['participant_code'].nunique()} participants")

# Create working copies (this is where we’ll add focal points + feature columns)
df_vizex_features = df_vizex.copy(deep=True)
df_vizex_targets_features = df_vizex_targets.copy(deep=True)

# Quick checks
display(df_vizex.head())
display(df_vizex_targets.head())

# Image roots (edit these if your on-disk folder names differ)
raw_data_path_vizex = raw_data_path                 # participant images
raw_data_path_targets_vizex = raw_data_path_targets # target images

image_paths = {
    "vizex": raw_data_path_vizex,
    "vizex_targets": raw_data_path_targets_vizex,
}

# Dataset registry
# - dataset_id: selects dataframe + image path
# - domain_key: selects preprocessing conventions (now just 'vizex')
datasets = {
    "vizex": {
        "name": "VizEx",
        "domain_key": "vizex",
        "df_raw": df_vizex,
        "df": df_vizex_features,          # working df
        "image_path": image_paths["vizex"],
    },
    "vizex_targets": {
        "name": "VizEx Targets",
        "domain_key": "vizex",            
        "df_raw": df_vizex_targets,
        "df": df_vizex_targets_features,
        "image_path": image_paths["vizex_targets"],
    },
}

# Sanity checks: required columns
REQUIRED_COLS = ["png_filename"]
for k, ds in datasets.items():
    missing = [c for c in REQUIRED_COLS if c not in ds["df"].columns]
    if missing:
        raise KeyError(f"Dataset '{k}' is missing required columns in df: {missing}")

# Sanity checks: image_path exists
for k, ds in datasets.items():
    if not os.path.isdir(ds["image_path"]):
        print(f"WARNING: image_path not found for '{k}': {ds['image_path']}")


In [ ]:
df_vizex

In [ ]:
def model_summary_string(mdf, var, precision=3):
    b = mdf.params[var]
    se = mdf.bse[var]
    p = mdf.pvalues[var]
    if p < 0.0001:
        p_str = "p < .0001"
    elif p < 0.001:
        # Avoid "p = .000"
        p_str = "p < .001"
    else:
        p_str = f"p = {p:.3f}".replace("0.", ".")
    return f"b = {b:.{precision}f}, SE = {se:.{precision}f}, {p_str}"

## Main experimental analysis

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import statsmodels.api as sm
import statsmodels.formula.api as smf
from statsmodels.stats.multitest import multipletests
import warnings
import os

# Only silence convergence chatter, not everything -- a blanket ignore hides the
# "Random effects covariance is singular" warnings that matter here.
warnings.filterwarnings("ignore", category=FutureWarning)
warnings.filterwarnings("ignore", category=DeprecationWarning)

# ============================================================
# GLOBAL ICONIC STYLING CONFIG
# ============================================================
LINE_WEIGHT_AXIS = 2.5      
LINE_WEIGHT_MEAN = 3.5      
FONT_SIZE_MAIN = 16         
FONT_SIZE_TICKS = 11        
MARKER_SIZE_DATA = 20       
MARKER_ALPHA = 0.12         
DEFAULT_COLOR = "#003366"   # Dark Blue
# ============================================================

# 1. Data Preparation
df = df_vizex.copy()
df = df[df['stimulus_type'] == 'LIGHT'].copy()

features = {
    'detail': 'detail level',
    'r_directionality': 'radial orientation',
    'theta_directionality': 'tangential orientation',
    'brightness': 'brightness',
    'contrast': 'contrast'
}

for col in list(features.keys()) + ['condition_hz']:
    df[col] = pd.to_numeric(df[col], errors='coerce')

# Clean floating point precision issues for categorical grouping
df['condition_hz'] = df['condition_hz'].round(2)
df = df.dropna(subset=list(features.keys()) + ['condition_hz', 'participant_code']).copy()

# Create categorical frequency column.
# NOTE: pd.Categorical(df['condition_hz'].sort_values()) drops the index and assigns
# POSITIONALLY, so every row got the i-th smallest Hz rather than its own. It was
# harmless only because nothing read hz_cat per-row -- only .cat.categories.
_hz_levels = sorted(df['condition_hz'].dropna().unique().tolist())
df['hz_cat'] = pd.Categorical(df['condition_hz'], categories=_hz_levels, ordered=True)
assert (df['hz_cat'].astype(float) == df['condition_hz']).all(), "hz_cat is misaligned"

# 2. Plotting Function
text_size = 15
main_title_size = int(text_size * 1.15)
labelpad = 14

def _fmt_num(num, decimals=1):
    return f"{float(num):.{decimals}f}"


def plot_feature_vs_frequency(feature: str, df: pd.DataFrame, *, 
                               feature_name: str, y_label: str, 
                               use_log_scale: bool = False, 
                               y_limit_percentile: float = None,
                               use_robust_avg: bool = False,
                               show_error_bars: bool = False,
                               force_y_min_0: bool = False,
                               line_color: str = DEFAULT_COLOR):
    rng = np.random.RandomState(42)
    jitter = 0.12
    hz_levels = list(df["hz_cat"].cat.categories)
    labels = [f"{_fmt_num(hz, 1).rstrip('0').rstrip('.')} Hz" for hz in hz_levels]
    x_pos = np.arange(len(hz_levels), dtype=float)

    fig_w = max(7.5, 0.6 * len(hz_levels))
    fig, ax = plt.subplots(figsize=(fig_w, 5.5))

    # --- Data Cleaning for Log Scale ---
    feature_data = df[feature].dropna()
    
    # If using log, we must ensure data is > 0
    if use_log_scale:
        # Shift data slightly if there are zeros or negatives
        if (feature_data <= 0).any():
            min_pos = feature_data[feature_data > 0].min() if (feature_data > 0).any() else 1e-5
            feature_data = feature_data.clip(lower=min_pos * 0.1)
            # Apply same clip to the dataframe for plotting
            df = df.copy()
            df[feature] = df[feature].clip(lower=min_pos * 0.1)

    # --- Improved Y-Limit Logic ---
    if y_limit_percentile is not None:
        margin = (100.0 - y_limit_percentile) / 2.0
        y_min, y_max = np.percentile(feature_data, [margin, 100.0 - margin])
        
        if use_log_scale:
            # Logarithmic buffer (multiplicative instead of additive)
            ax.set_ylim(y_min * 0.8, y_max * 1.2)
        else:
            y_range = y_max - y_min
            ax.set_ylim(y_min - (0.02 * y_range), y_max + (0.02 * y_range))
    
    # Force Y-Min to 0 ONLY if not in log scale
    if force_y_min_0 and not use_log_scale:
        ax.set_ylim(bottom=0)

    # --- Raw Data Points ---
    for i, hz in enumerate(hz_levels):
        vals = df.loc[df["condition_hz"] == hz, feature].to_numpy(dtype=float)
        if vals.size == 0: continue
        xs = x_pos[i] + rng.normal(0, jitter, size=vals.size)
        ax.scatter(xs, vals, s=MARKER_SIZE_DATA, color="black", 
                   alpha=MARKER_ALPHA, edgecolors='none', zorder=1, clip_on=False)

    # --- Statistics Calculation ---
    grouped = df.groupby("condition_hz")[feature]
    if use_robust_avg:
        center_vals = grouped.median().reindex(hz_levels).to_numpy(dtype=float)
    else:
        center_vals = grouped.mean().reindex(hz_levels).to_numpy(dtype=float)
    
    if show_error_bars:
        err_vals = grouped.sem().reindex(hz_levels).to_numpy(dtype=float)
        ax.errorbar(x_pos, center_vals, yerr=err_vals, fmt='none', 
                    ecolor=line_color, elinewidth=LINE_WEIGHT_AXIS, 
                    capsize=4, zorder=2, clip_on=False)

    # Bold Central Line
    ax.plot(x_pos, center_vals, linewidth=LINE_WEIGHT_MEAN, color=line_color, marker='o', 
            markersize=7, markerfacecolor='white', markeredgewidth=2, zorder=3, clip_on=False)

    # --- Styling ---
    if use_log_scale: 
        ax.set_yscale('log')
        from matplotlib.ticker import ScalarFormatter
        ax.yaxis.set_major_formatter(ScalarFormatter())

    for s in ['top', 'right']: ax.spines[s].set_visible(False)
    for s in ['left', 'bottom']: ax.spines[s].set_linewidth(LINE_WEIGHT_AXIS)

    ax.set_title(f"{feature_name} vs stroboscopic frequency", fontsize=FONT_SIZE_MAIN, fontweight='bold', loc='left', pad=15)
    ax.set_ylabel(y_label, fontsize=FONT_SIZE_MAIN, fontweight='bold', labelpad=12)
    ax.set_xlabel("stroboscopic frequency", fontsize=FONT_SIZE_MAIN, fontweight='bold', labelpad=12)

    ax.set_xticks(x_pos)
    ax.set_xticklabels(labels, rotation=45, ha='right', fontsize=FONT_SIZE_TICKS, fontweight='bold')
    ax.tick_params(axis="both", width=LINE_WEIGHT_AXIS, labelsize=FONT_SIZE_TICKS)

    plt.tight_layout()
    plt.show()

# 3. Modeling and Statistics (Optimized for Variance Reporting)
results_list = []
variance_export_list = [] # Data sink for signal/variance visualization

for feat, name in features.items():
    plot_feature_vs_frequency(feat, df, feature_name=name, y_label=name)
    
    formula = f"{feat} ~ C(condition_hz)"
    try:
        model = smf.mixedlm(formula, df, groups=df["participant_code"])
        result = model.fit(reml=True)
        
        wald_test = result.wald_test_terms()
        term_name = "C(condition_hz)"
        
        # Core Statistics
        chi2_val = float(np.squeeze(wald_test.table.loc[term_name, 'statistic']))
        df_num = int(np.squeeze(wald_test.table.loc[term_name, 'df_constraint']))
        p_val_uncorr = float(np.squeeze(wald_test.table.loc[term_name, 'pvalue']))
        
        # Variance Components
        var_resid = result.scale
        sd_resid = np.sqrt(var_resid) 
        var_participant = result.cov_re.iloc[0, 0]
        icc = var_participant / (var_participant + var_resid)
        
        # Signal-to-Noise Ratio
        grand_mean = df[feat].mean()
        trial_cv = (sd_resid / grand_mean) * 100 if grand_mean != 0 else 0

        results_list.append({
            'Feature': name,
            'Mean': grand_mean,
            'Chi2': chi2_val,
            'df': df_num,
            'p_unc': p_val_uncorr,
            'Trial_SD': sd_resid,
            'Trial_CV%': trial_cv,
            'ICC': icc
        })
        
        # Export metrics for visualization 
        fixed_predict = model.predict(result.params)
        signal_var = float(np.var(fixed_predict))
        
        re_singular = bool(not np.isfinite(var_participant) or var_participant < 1e-10)
        if re_singular:
            print(f"  WARNING [{name}]: random-effects variance ~0 "
                  f"({var_participant:.3g}); ICC and DT_plus_HT are unidentified.")

        variance_export_list.append({
            'Feature': feat,
            'Signal_Exp': signal_var,
            'DT_plus_HT': var_participant,
            'DN_plus_HN': var_resid,
            'RE_singular': re_singular
        })
        
    except Exception as e:
        print(f"Failed to fit model for {name}: {e}")

# Save the raw variance arrays for the 3rd script
os.makedirs("../015_tables", exist_ok=True)
pd.DataFrame(variance_export_list).to_csv("../015_tables/experimental_results_full.csv", index=False)

# 4. Correction and Output Formatting
res_df = pd.DataFrame(results_list)
reject, p_holm, _, _ = multipletests(res_df['p_unc'], alpha=0.05, method='holm')
res_df['p_holm'] = p_holm

print("\n\n" + "="*85)
print("STATISTICAL SUMMARY: TRIAL-LEVEL NOISE VS. TREND SIGNIFICANCE")
print("="*85)

disp = res_df.copy()
disp['Mean'] = disp['Mean'].round(2)
disp['Chi2'] = disp['Chi2'].round(1)
disp['Trial_SD'] = disp['Trial_SD'].round(2)
disp['Trial_CV%'] = disp['Trial_CV%'].map("{:.1f}%".format)
disp['ICC'] = disp['ICC'].round(3)
disp['p_holm'] = disp['p_holm'].apply(lambda x: "< .001" if x < 0.001 else f"{x:.4f}")

cols = ['Feature', 'Mean', 'Chi2', 'df', 'p_holm', 'Trial_SD', 'Trial_CV%', 'ICC']
print(disp[cols].to_string(index=False))

print("\nInterpretation Note: High Trial_SD with low ICC indicates variance is driven by")
print("stochastic trial-level noise, not stable participant differences.")

## extra 15 hz test

In [ ]:
import numpy as np
import pandas as pd
import statsmodels.formula.api as smf
import matplotlib.pyplot as plt
from scipy.stats import norm
import warnings

warnings.filterwarnings("ignore")

# ============================================================
# GLOBAL ICONIC STYLING CONFIG
# ============================================================
LINE_WEIGHT_AXIS = 2.5      
LINE_WEIGHT_MEAN = 3.5      
FONT_SIZE_MAIN = 16         
FONT_SIZE_TICKS = 11        
MARKER_SIZE_DATA = 20       
MARKER_ALPHA = 0.12         
DEFAULT_COLOR = "#003366"

# ============================================================
# 10 Hz vs 15 Hz Analysis (Trial-Level LME Only)
# ============================================================

def run_detail_10_vs_15():
    # 1. Data Cleaning and Preparation
    df = df_vizex.copy()
    df = df[df['stimulus_type'] == 'LIGHT'].copy()
    df['condition_hz'] = pd.to_numeric(df['condition_hz'], errors='coerce')
    df['detail'] = pd.to_numeric(df['detail'], errors='coerce')
    
    # Isolate conditions of interest
    df = df[df["condition_hz"].isin([10, 15])].copy()
    df = df.dropna(subset=['detail', 'condition_hz', 'participant_code']).copy()
    
    # Categorical enforcement for coefficient estimation
    df["hz_cat"] = pd.Categorical(df["condition_hz"], categories=[10, 15], ordered=True)

    print(f"Total Trials: {len(df)}")
    print(f"Unique Participants: {df['participant_code'].nunique()}")

    # 2. Linear Mixed-Effects Model
    # Note: 10Hz is the intercept; coefficient reflects (15Hz - 10Hz)
    model = smf.mixedlm("detail ~ C(hz_cat)", df, groups=df["participant_code"])
    m = model.fit(reml=False)#, method="lbfgs")

    print("\n================ LME RESULTS (10 Hz vs 15 Hz) ================\n")
    print(m.summary())

    # 3. APA String Extraction
    # Handling version-specific coefficient naming in statsmodels
    coef_key = [k for k in m.params.keys() if "T.15" in k]
    
    if coef_key:
        k = coef_key[0]
        b = m.params[k]
        se = m.bse[k]
        z = b / se
        p = 2 * norm.sf(abs(z))
        
        p_str = "p < .001" if p < 0.001 else f"p = {p:.3f}".replace("0.", ".", 1)
        report = f"(z = {z:.2f}, {p_str})"
        print(f"\nAPA Reporting Fragment (Detail):\n{report}\n")

    # 4. Visualization
    plot_feature_vs_frequency(
        feature="detail",
        df=df,
        feature_name="Detail Level",
        y_label="Normalised detail level",
        show_error_bars=True
    )

# ---------------- RUN ----------------
run_detail_10_vs_15()